# Packaging, environments and performance
* we already covered the very basics of pip and virtual environments in `00_start_development_things_todo.md` (venv create/activate, pip install/freeze) - go read that again quickly if its been a while, we wont repeat it here
* in this lesson we go deeper into how real python projects describe their dependencies (`requirements.txt` limits, and the modern `pyproject.toml` standard), touch on modern all-in-one tools like Poetry and uv, and then switch gears into measuring and improving performance with `timeit` and `cProfile`
* packaging and performance dont look related at first glance, but both are about "how do i ship/run my code properly", so they fit into one lesson together

## requirements.txt recap and its limitations
* a `requirements.txt` is just a plain text file, one package (optionally pinned to a version) per line
* `pip freeze > requirements.txt` dumps every installed package with its exact version - simple, but it has real limitations:
  * no built-in separation of "dev dependencies" (pytest, linters, ...) vs "production dependencies" (what your app actually needs to run) - people work around this with multiple files like `requirements-dev.txt`, but its a convention, not a language feature
  * its not a real lock file - `pip freeze` only captures whats currently installed, it doesnt guarantee anyone else resolves the exact same transitive dependency tree again in the future (a sub-dependency might get a new compatible release)
  * it carries zero project metadata - no project name, version, author, python-version requirement, entry points, ... its ONLY a list of packages
* these gaps are exactly why the python ecosystem moved towards a single standardized project description file

## pyproject.toml - the modern standard (PEP 621)
* `pyproject.toml` is now the recommended, standardized place to describe a python project: metadata (name, version, description, authors) AND dependencies, all in one file
* its written in TOML format (a simple, readable config format - think of it as a stricter, typed cousin of an `.ini` file)
* below is an EXAMPLE `pyproject.toml`, we are not executing this (its not python code, its a config file), just look at the shape of it

```toml
[project]
name = "mytools"
version = "0.1.0"
description = "a small collection of helper tools"
authors = [{ name = "Alex", email = "alex@example.com" }]
requires-python = ">=3.9"
dependencies = [
    "requests>=2.28,<3.0",
    "click~=8.1",
]

[project.optional-dependencies]
dev = ["pytest>=7.0", "black"]

[build-system]
requires = ["setuptools>=61.0"]
build-backend = "setuptools.build_meta"
```
* the `[project]` table holds name/version/dependencies - this is the part every tool (pip, poetry, uv, build backends) agrees on reading, thats the whole point of PEP 621 standardizing it
* `[project.optional-dependencies]` finally gives us that dev-vs-prod separation `requirements.txt` was missing - install extras with `pip install ".[dev]"`
* `[build-system]` tells tools HOW to actually build a distributable package out of your source code (not needed for a runnable app you dont plan to publish, but needed for a real installable python library)

## semantic versioning and version specifiers
* most python packages (and packages in basically every other ecosystem) follow semantic versioning: `MAJOR.MINOR.PATCH`, e.g. `2.15.3`
  * `MAJOR` increases on breaking/incompatible changes
  * `MINOR` increases when new features are added in a backwards-compatible way
  * `PATCH` increases for backwards-compatible bugfixes
* both `requirements.txt` and `pyproject.toml` use the same version specifier syntax to constrain which versions are acceptable:
  * `requests==2.31.0` - exactly this version, nothing else
  * `requests>=2.0` - this version or newer, no upper bound (a bit risky, a future breaking major could sneak in)
  * `requests>=1.2,<2.0` - a range, common way to allow minor/patch updates but block the next breaking major
  * `requests~=1.2` - "compatible release" operator, roughly means ">=1.2, <2.0" (allows minor/patch updates within the same major version) - very similar in spirit to `^1.2.0` in npm's package.json

## modern tooling: Poetry and uv (conceptual overview, we dont install them here)
* the traditional python workflow is really "compose it yourself": `venv` for isolation + `pip` for installing + a `requirements.txt`/`pyproject.toml` for tracking - several small, separate stdlib-adjacent tools you glue together
* **Poetry** and **uv** are popular modern tools that instead try to be one all-in-one tool: they manage the virtual environment, install dependencies, generate a real lock file (exact resolved versions of everything, including transitive dependencies) AND build/publish your package, all through one CLI and one `pyproject.toml`
* uv (from the makers of ruff) is especially known for being extremely fast (written in Rust) and is quickly becoming a very popular drop-in replacement for pip+venv+pip-tools
* philosophy difference: pip+venv+requirements.txt is "many small unix-philosophy tools, you wire them together yourself, full control but more manual steps" - Poetry/uv are "one opinionated tool handles the whole lifecycle, less setup but you buy into their way of doing things"
* we wont run poetry/uv commands in this notebook (they need to be installed separately and are out of scope for this lesson) - just know they exist and why people reach for them

## editable installs: `pip install -e .`
* when you are developing your OWN package (one that has a `pyproject.toml`/`setup.py`), a normal `pip install .` COPIES your code into the environment's site-packages folder - if you then edit your source code, the installed copy does NOT change, you would need to reinstall every time
* `pip install -e .` (the `-e` stands for "editable") instead installs a link back to your source folder - python imports your package straight from where your source code lives
* this means: you edit a `.py` file in your package, and the change is immediately visible the next time you import it, no reinstall needed
* extremely useful while actively developing a package you intend to publish or reuse across multiple projects on your own machine

## measuring performance with timeit
* before optimizing anything, we should MEASURE - guessing what is "slow" is often wrong
* the `timeit` module runs a small piece of code many times and measures how long it takes, its designed to avoid common pitfalls of naive manual timing (like other background processes messing with a single measurement)
* lets compare 3 ways of building the same list of squares: a list comprehension, a classic for-loop with `.append()`, and `map()`

In [1]:
import timeit

# we keep the input small and use a modest "number=" (repeat count) so this cell runs fast
# the goal here is NOT to get a super precise benchmark, its to see the RELATIVE difference between approaches
data = range(1000)  # 1000 numbers, small enough to run fast, big enough to see a difference

# variant 1: list comprehension
def via_comprehension():
    return [n ** 2 for n in data]

# variant 2: classic for-loop with .append()
def via_for_loop():
    result = []
    for n in data:
        result.append(n ** 2)  # each .append() call has function-call overhead, comprehensions avoid a lot of that
    return result

# variant 3: map() with a lambda
def via_map():
    return list(map(lambda n: n ** 2, data))  # map() is lazy, we wrap it in list() to actually build the list

# timeit.timeit(function, number=...) calls "function" that many times and returns the TOTAL time in seconds
runs = 500  # how often each variant gets executed - modest so the whole cell finishes quickly
time_comprehension = timeit.timeit(via_comprehension, number=runs)
time_for_loop = timeit.timeit(via_for_loop, number=runs)
time_map = timeit.timeit(via_map, number=runs)

print(f"list comprehension: {time_comprehension:.4f} seconds for {runs} runs")
print(f"for-loop + append:  {time_for_loop:.4f} seconds for {runs} runs")
print(f"map() + lambda:     {time_map:.4f} seconds for {runs} runs")

# IMPORTANT: the exact numbers above depend heavily on your machine, current load, python version etc - dont trust absolute seconds
# what matters pedagogically is the RELATIVE ordering and WHY: comprehensions loop in optimized C code inside the interpreter,
# a manual for-loop pays extra python-level overhead for every single .append() call, and map()+lambda adds function-call overhead per item too

list comprehension: 0.0175 seconds for 500 runs
for-loop + append:  0.0188 seconds for 500 runs
map() + lambda:     0.0268 seconds for 500 runs


## profiling with cProfile
* `timeit` is great for comparing a couple of small alternative snippets, but for a bigger function we usually want to know WHERE inside it the time actually goes
* `cProfile` is a builtin profiler that records how many times every function was called and how long each one took
* lets deliberately write an inefficient function: checking "is this number in a list?" inside a loop, list membership checks are O(n) (has to scan every item), compared to a set which is O(1) (direct hash lookup)

In [2]:
import cProfile
import pstats
import io

def build_lookup_as_list(size):
    # a plain list of numbers - membership checks against this will be slow (O(n) per check)
    return list(range(size))

def build_lookup_as_set(size):
    # the same numbers, but as a set - membership checks against this are fast (O(1) per check)
    return set(range(size))

def slow_membership_checks(numbers_to_check, lookup_list):
    # this scans "lookup_list" from the start every single time - gets slower the bigger lookup_list is
    found = 0
    for number in numbers_to_check:
        if number in lookup_list:  # O(n) list scan - this is the deliberately slow part
            found += 1
    return found

def fast_membership_checks(numbers_to_check, lookup_set):
    # same logic, but "in" on a set is a hash lookup instead of a scan
    found = 0
    for number in numbers_to_check:
        if number in lookup_set:  # O(1) hash lookup - this is the fast part
            found += 1
    return found

def do_the_work():
    # this is the function we profile - it calls both the slow and the fast variant so we can compare them in the stats
    lookup_list = build_lookup_as_list(20000)   # a fairly big list, so the O(n) cost is actually visible
    lookup_set = build_lookup_as_set(20000)     # the same data as a set
    numbers_to_check = range(0, 20000, 7)       # a handful of numbers we look up repeatedly

    slow_membership_checks(numbers_to_check, lookup_list)  # the slow O(n)-per-check version
    fast_membership_checks(numbers_to_check, lookup_set)    # the fast O(1)-per-check version

# cProfile.Profile() lets us profile just the block we care about, instead of the whole script
profiler = cProfile.Profile()
profiler.enable()   # start recording
do_the_work()        # the code we want to measure
profiler.disable()  # stop recording

# pstats.Stats formats the recorded profiling data, we sort by "cumulative" time (time in a function PLUS everything it called)
stats_stream = io.StringIO()  # we print into a string buffer so notebook output stays tidy and short
stats = pstats.Stats(profiler, stream=stats_stream).sort_stats("cumulative")
stats.print_stats(6)  # only show the top 6 lines, otherwise the output gets very long

print(stats_stream.getvalue())

# how to read this output:
# ncalls   = how often the function was called
# tottime  = time spent INSIDE that function itself, excluding calls to other functions
# cumtime  = total time including everything that function called (cumulative) - this is what we sorted by
# looking at "slow_membership_checks" vs "fast_membership_checks" here, the slow list-based one shows a clearly higher cumtime,
# exactly because "in" on a list has to scan, while "in" on a set is a near-instant hash lookup

         47 function calls in 0.155 seconds

   Ordered by: cumulative time
   List reduced from 24 to 6 due to restriction <6>

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
        2    0.000    0.000    0.155    0.078 /usr/local/lib/python3.11/dist-packages/IPython/core/interactiveshell.py:3786(run_code)
        2    0.000    0.000    0.155    0.078 {built-in method builtins.exec}
        1    0.000    0.000    0.154    0.154 /tmp/ipykernel_3361/1092934559.py:29(do_the_work)
        1    0.151    0.151    0.151    0.151 /tmp/ipykernel_3361/1092934559.py:13(slow_membership_checks)
        1    0.002    0.002    0.002    0.002 /tmp/ipykernel_3361/1092934559.py:9(build_lookup_as_set)
        1    0.001    0.001    0.001    0.001 /tmp/ipykernel_3361/1092934559.py:5(build_lookup_as_list)





## practical performance tips
* prefer built-in functions and comprehensions over manual python-level loops where reasonable - a lot of them (like `sum()`, `map()`, list/dict/set comprehensions) run their looping in optimized C code inside the interpreter
* use a `set` or `dict` instead of a `list` when you need to repeatedly check "is this value in there?" - O(1) average lookup instead of O(n), we literally just saw this difference in the cProfile output above
* avoid premature optimization - "measure first, optimize what's actually slow" - dont waste time hand-optimizing a function that only runs once and takes 2 milliseconds, profile first to find the ACTUAL bottleneck
* callback to lesson 11 (concurrency): pure python code is still bound by the GIL (Global Interpreter Lock), so for real CPU-bound speedups (not just "slightly nicer code") you eventually need multiprocessing or native extensions (numpy, C-extensions), not just clever pure-python tricks

## a quick word on memory
* CPython manages memory for us automatically: every object has a reference count (how many places point to it), and once that count hits zero the memory gets freed - theres also a cyclic garbage collector (module `gc`) that cleans up reference cycles the simple refcounting alone cant catch
* this means we almost NEVER manage memory manually in python (no manual `malloc`/`free` like C/C++, and no compile-time ownership/borrow-checking like Rust) - convenient, but it also means memory usage can be a bit less predictable/visible than in those languages
* `sys.getsizeof()` gives us the size (in bytes) of a single object in memory - lets make memory usage a bit more tangible

In [3]:
import sys

# lets look at the memory footprint of a few everyday objects
an_integer = 42
a_string = "hello world"
a_list = [1, 2, 3, 4, 5]

print("size of an int (42):        ", sys.getsizeof(an_integer), "bytes")
print("size of a string:           ", sys.getsizeof(a_string), "bytes")
print("size of a list [1..5]:      ", sys.getsizeof(a_list), "bytes")

# fun fact: even a "small" python object carries some overhead (type info, refcount, ...) compared to a raw C int/char array
# thats part of the tradeoff for pythons dynamic, automatically-managed objects vs a lower-level languages raw memory layout

size of an int (42):         28 bytes
size of a string:            60 bytes
size of a list [1..5]:       104 bytes


## quick comparison to other languages
* python's `pip` + `requirements.txt`/`pyproject.toml` is comparable to JavaScript's `npm` + `package.json`/`package-lock.json`, Java's Maven/Gradle (`pom.xml`/`build.gradle`), C#'s NuGet (package references in the `.csproj`), and Go's `go.mod`/`go.sum`
* Rust's `cargo` + `Cargo.toml`/`Cargo.lock` is widely considered the gold standard here - ONE official tool that does dependency management, building, testing AND publishing, with a smooth built-in lock file from day one
* unlike cargo (Rust) or npm (JavaScript), which each ship one official all-in-one tool for their ecosystem, python historically had a much more fragmented landscape: pip, virtualenv, setuptools, Poetry, Pipenv, conda, uv, ... all doing overlapping jobs slightly differently - the modern `pyproject.toml` standard (PEP 621) is pythons attempt to unify around one shared, tool-agnostic project description format
* memory management: python (refcounting + cyclic GC) is in the same broad "garbage collected" category as Java, C#, JavaScript and Go - the runtime frees memory for you automatically - but its fundamentally different from C/C++ (fully manual `malloc`/`free`) and from Rust (memory safety enforced at COMPILE time via ownership/borrowing, no runtime garbage collector needed at all)

## Exercises
1. write a tiny example `pyproject.toml` `[project]` section as a markdown code block (```toml ... ```) for a fictional package called "mytools", version "0.1.0", depending on `requests>=2.0`
2. use `timeit` to compare summing a list of 1000 numbers with a plain for-loop (`total += n`) vs the builtin `sum()` function
3. use `cProfile` on a small function that calls two other functions, then look at the printed stats and identify which of the two nested calls has the higher cumulative time
4. rewrite a slow "is this value in this big list?" check as a set-based check instead, and briefly explain (in a comment) why the set version is algorithmically faster - a modest-sized demo is enough, no need for a huge benchmark

In [4]:
# TODO: solve exercise 1 here (write the pyproject.toml [project] section as a comment or a string, its not runnable toml anyway)


# TODO: solve exercise 2 here (timeit: for-loop sum vs builtin sum())


# TODO: solve exercise 3 here (cProfile on a function calling two nested functions)


# TODO: solve exercise 4 here (list-based check vs set-based check)

### sample solutions
* try the exercises above yourself first, only look here if you are stuck or want to compare your approach

In [5]:
import timeit
import cProfile
import pstats
import io

# sample solution 1 - example pyproject.toml [project] section for "mytools"
pyproject_example = '''
[project]
name = "mytools"
version = "0.1.0"
dependencies = [
    "requests>=2.0",
]
'''
print(pyproject_example)

# sample solution 2 - for-loop sum vs builtin sum()
numbers = list(range(1000))  # our test data, 1000 numbers

def sum_with_for_loop():
    total = 0
    for n in numbers:  # manual accumulation, python-level loop overhead on every iteration
        total += n
    return total

def sum_with_builtin():
    return sum(numbers)  # sum() loops in optimized C code internally

runs = 1000
time_for_loop_sum = timeit.timeit(sum_with_for_loop, number=runs)
time_builtin_sum = timeit.timeit(sum_with_builtin, number=runs)
print(f"for-loop sum:   {time_for_loop_sum:.4f} seconds for {runs} runs")
print(f"builtin sum():  {time_builtin_sum:.4f} seconds for {runs} runs")
# again: absolute numbers vary by machine, but sum() being faster or at least competitive is the expected pattern,
# because it avoids python-level loop overhead by doing the accumulation in C

# sample solution 3 - cProfile on a function with two nested calls
def quick_helper():
    return sum(range(100))  # a cheap nested call

def slow_helper():
    return sum(n * n for n in range(200000))  # a much more expensive nested call, on purpose

def outer_function():
    quick_helper()
    slow_helper()

profiler = cProfile.Profile()
profiler.enable()
outer_function()
profiler.disable()

buffer = io.StringIO()
stats = pstats.Stats(profiler, stream=buffer).sort_stats("cumulative")
stats.print_stats(6)
print(buffer.getvalue())
# reading the stats: "slow_helper" shows a clearly higher cumtime than "quick_helper",
# because it iterates over 200000 items instead of just 100 - thats the nested call actually eating our time budget

# sample solution 4 - list-based vs set-based membership check
big_list = list(range(50000))   # membership check against this is O(n) - has to scan
big_set = set(range(50000))      # membership check against this is O(1) average - direct hash lookup

value_to_find = 49999  # deliberately near the end, worst case for the list scan

is_in_list = value_to_find in big_list  # slow path: python scans up to all 50000 items one by one
is_in_set = value_to_find in big_set    # fast path: python computes a hash and jumps straight to the bucket

print("found in list:", is_in_list)
print("found in set:", is_in_set)
# WHY the set is algorithmically faster: a list membership check ("in") has to potentially compare against EVERY item
# until it finds a match (O(n), worst case scans the whole list), while a set stores items in a hash table and can
# jump almost directly to where the value would be (O(1) on average) - the bigger the collection, the bigger the gap gets


[project]
name = "mytools"
version = "0.1.0"
dependencies = [
    "requests>=2.0",
]



for-loop sum:   0.0208 seconds for 1000 runs
builtin sum():  0.0044 seconds for 1000 runs
         200048 function calls in 0.041 seconds

   Ordered by: cumulative time
   List reduced from 24 to 6 due to restriction <6>

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
        2    0.000    0.000    0.040    0.020 /usr/local/lib/python3.11/dist-packages/IPython/core/interactiveshell.py:3786(run_code)
        2    0.000    0.000    0.040    0.020 {built-in method builtins.exec}
        1    0.000    0.000    0.040    0.040 /tmp/ipykernel_3361/470241838.py:44(outer_function)
        1    0.000    0.000    0.040    0.040 /tmp/ipykernel_3361/470241838.py:41(slow_helper)
        2    0.019    0.010    0.040    0.020 {built-in method builtins.sum}
   200001    0.021    0.000    0.021    0.000 /tmp/ipykernel_3361/470241838.py:42(<genexpr>)



found in list: True
found in set: True


* congratulation you finished this lesson

## what we learned
* the limitations of a plain `requirements.txt` (no dev/prod split, no real lock file, no project metadata) and how `pyproject.toml` (PEP 621) addresses them with a standardized `[project]` table
* semantic versioning (`MAJOR.MINOR.PATCH`) and version specifiers (`==`, `>=`, ranges, `~=`) used in both requirements files and pyproject.toml
* the philosophy difference between composing traditional tools (pip+venv+requirements.txt) and modern all-in-one tools (Poetry, uv)
* editable installs (`pip install -e .`) and why they matter while developing your own package
* measuring performance honestly with `timeit` (comparing comprehension vs for-loop vs map()) and profiling with `cProfile`/`pstats` to find where time actually goes
* practical performance habits: prefer builtins/comprehensions, use sets/dicts for membership checks, measure before optimizing, and remember the GIL limits pure-python CPU-bound speedups
* a first look at memory with `sys.getsizeof()` and why python rarely needs manual memory management (refcounting + the `gc` module) compared to C/C++ or Rust